# GitHub & Remote Collaboration
**Topic:** Git & Version Control

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** the relationship between a local Git repository and a remote repository on GitHub
- **Describe** what each of `clone`, `push`, `pull`, and `fetch` does to the local and remote repository
- **Interpret** the pull request workflow as a structured code review process before merging

> **Tip:** Start by examining the **local vs remote flow diagram** and trace one complete round trip: you make a commit locally, push it to GitHub, a teammate pulls it, makes a change, and pushes back. Follow that arrow chain before reading the explanation.

---
## How we got here

In *03: Branching & Merging* we learned to work in parallel on local branches. Everything so far has happened on your own computer. GitHub is where your local repository gets backed up, shared with teammates, and opened up for review. This notebook connects your local Git workflow to the internet and to other people, which is where Git becomes genuinely powerful.

---
## Why this matters for data science

GitHub is the portfolio, the collaboration platform, and the deployment trigger for data science work. Your GitHub profile is often the first thing a technical interviewer looks at. Pull requests are how teams review code, catch bugs before they reach production, and maintain a record of why decisions were made. Knowing how to clone a repo, push a branch, and open a pull request is a day-one expectation at every data science team that works professionally.

---
## Try it yourself

In [ ]:
# Widget: A pull request lifecycle simulator with two panels
# Left panel: "Local Repository" — shows branches and commits
# Right panel: "GitHub (Remote)" — shows the remote repo state
# Step-through buttons at the bottom:
#   Step 1: Clone — remote repo appears in local panel (with origin/main pointer)
#   Step 2: Create branch — new branch appears locally only
#   Step 3: Make a commit — new commit appears on local branch
#   Step 4: Push — the branch appears on the remote panel as well
#   Step 5: Open PR — a pull request card appears on the remote panel
#   Step 6: Review & Approve — a green checkmark appears on the PR card
#   Step 7: Merge PR — the branch merges into remote main; local main is now behind
#   Step 8: Pull — local main advances to match remote main
# Student should notice: local and remote are separate; push and pull are explicit sync steps;
#   the PR lives entirely on GitHub, not in Git itself

---
## What's happening?

A remote repository is a copy of your Git repository hosted on a server (GitHub, GitLab, Bitbucket). Local and remote are kept in sync manually through explicit commands.

```
LOCAL MACHINE                         GITHUB (REMOTE)
─────────────────────────────────     ──────────────────────────────
Your working directory                 origin/main (the remote copy)
        │                                        │
  git add + git commit                           │
        │                                        │
  git push ──────────────────────────────────────►
        │                                        │
        ◄────────────────────────────────── git pull
        │                                        │
  git fetch ─────────────────────────────────────►
  (download, don't merge)
```

| Command | What it does to local | What it does to remote |
|---------|----------------------|----------------------|
| `git clone <url>` | Creates a full local copy of the remote repo | Nothing: read-only operation |
| `git push` | Uploads local commits to the remote branch | Updates the remote branch to match local |
| `git pull` | Downloads remote commits and merges into local branch | Nothing: read-only on remote |
| `git fetch` | Downloads remote commits without merging | Nothing: read-only on remote |
| `git push -u origin branch` | Pushes a new branch for the first time and sets tracking | Creates the branch on GitHub |

### The pull request workflow

A pull request (PR) is not a Git concept; it is a GitHub feature. It is a formal request to merge one branch into another, with a built-in code review interface.

```
1. Create a branch locally
2. Make commits on that branch
3. Push the branch to GitHub (git push -u origin my-branch)
4. Open a pull request on GitHub (base: main ← compare: my-branch)
5. Teammates review the code, leave comments, request changes
6. You address comments with additional commits on the same branch
7. Reviewer approves; the PR is merged into main
8. Delete the branch on GitHub and locally
```

Return to the simulator and click through all 8 steps to see how local and remote state change at each one.

---
## Real-world example: A data science team collaborating on a shared model repository

The chart shows a 30-day collaboration graph for a four-person data science team. Each bar represents one team member's contributions; the segments show commits, pull requests opened, and pull requests reviewed.

Notice:

- **Notice:** Code reviews (PR reviews) are distributed across the team, not just done by the lead; this is the professional norm because diverse reviewers catch different types of problems
- **Notice:** The most productive team member by commit count is not the same as the most valuable by PR reviews; reviewing others' code is a meaningful contribution that does not show up in commit counts
- **Notice:** Pull requests serve as documentation: months later the PR description and review comments explain *why* a decision was made, which is information that does not fit in a commit message

> **Discussion question:** Your team has been committing directly to main for a month "to move faster." Today, a bad commit reached production and caused an incident. A teammate proposes switching to pull requests for all changes. What would you lose in speed, and what would you gain? How would you structure the PR process to minimize the speed cost?

In [ ]:
# ── Team collaboration graph: 30 days on a shared DS model repo ──────────────
team = ["Alice (lead)", "Bob", "Carol", "David"]
commits      = [28, 19, 23, 15]
prs_opened   = [8,  6,  9,  5]
prs_reviewed = [12, 14, 8, 11]

x = np.arange(len(team))
width = 0.25

fig = go.Figure()
for vals, name, color in [
    (commits,      "Commits",        PALETTE["primary"]),
    (prs_opened,   "PRs Opened",     PALETTE["secondary"]),
    (prs_reviewed, "PRs Reviewed",   PALETTE["accent"]),
]:
    fig.add_trace(go.Bar(
        x=[xi + width * (list(zip(*[(commits, prs_opened, prs_reviewed)])
                              ).index(vals) - 1) for xi in x],
        y=vals,
        name=name, marker_color=color,
        text=vals, textposition="outside",
    ))
fig.update_xaxes(tickvals=list(x), ticktext=team)
layout = base_layout(
    title="Team Collaboration — 30-Day Shared Model Repository",
    xaxis_title="Team Member",
    yaxis_title="Count",
)
layout.update(barmode="group")
fig.update_layout(layout)
fig.show()

### Local vs remote operation reference

| Action | Command | What it does to local | What it does to remote |
|--------|---------|----------------------|----------------------|
| Download a repo | `git clone <url>` | Creates full local copy | Nothing |
| Send commits to GitHub | `git push` | Nothing | Updates remote branch |
| Get teammate's commits | `git pull` | Merges remote changes into local branch | Nothing |
| Download without merging | `git fetch` | Updates remote-tracking branches only | Nothing |
| Create remote branch | `git push -u origin branch` | Sets tracking relationship | Creates branch on GitHub |
| Open a pull request | (GitHub UI or `gh pr create`) | Nothing | Creates a PR on GitHub |
| Merge a PR | (GitHub UI or `gh pr merge`) | Nothing until you `git pull` | Merges branch into target on GitHub |
| Delete remote branch | `git push origin --delete branch` | Nothing | Removes branch from GitHub |

---
## Key takeaway

> **GitHub is a remote copy of your repository; push and pull are the explicit sync steps that keep local and remote in agreement, and pull requests are the structured conversation layer that makes team collaboration safe and auditable.**

---
*You have completed the Git series. You now have the version control foundation that every professional data science role expects — commit often, branch always, and review before merging.*